# Strings — Interview Masterclass

The third notebook in the DSA series. Strings are essentially arrays of characters, so many array patterns transfer directly — but strings also have their own families of problems (palindromes, pattern matching, parsing, tries) that have no array analog.

Out of scope by design: binary search, sorting algorithms, recursion, backtracking, dynamic programming (string DP — LCS, edit distance, regex matching — is huge but lives in the DP notebook).

## How to use this notebook
1. **Section 1** — internalize Python string fundamentals. Immutability and slicing are the #1 source of bugs.
2. **Section 2** — method reference. Strings have ~45 methods; you don't need all of them, but the 15 below are non-negotiable.
3. **Sections 3-13** — the pattern families, with worked examples and LeetCode practice lists.
4. **Section 14** — cheat sheet and interview narration.

## What patterns cover ~90% of string interviews
Five families:
1. **Two pointers** — palindromes, reversal, in-place manipulation
2. **Sliding window** — substring with property X, anagrams in a string
3. **Character frequency / hashing** — anagram family, isomorphic, group by signature
4. **Pattern matching** — KMP, Rabin-Karp, Z-function
5. **Trie / prefix tree** — autocomplete, prefix search, word dictionaries

Plus three smaller-but-common families:
6. **Palindrome family** — expand-around-center, Manacher (very rare)
7. **Parsing / manipulation** — atoi, Roman, integer-to-words
8. **Encoding** — run-length, custom serialization, base conversion

---
# 1. Python string fundamentals

## Immutability — the most important thing
Python `str` objects are **immutable**. Every "modification" creates a new string.

```python
s = "hello"
s[0] = 'H'        # TypeError: 'str' object does not support item assignment
s = 'H' + s[1:]   # OK — but allocates a new string each time
```

**Why this matters in interviews:** building a string char by char with `result += c` is **O(n²)** because each `+=` allocates a new string and copies the old contents. The Pythonic fixes:
- Use a `list` and `''.join(list)` at the end — O(n)
- Use `io.StringIO` for very large strings
- Slice or use string methods rather than character-by-character concat

## Complexity reference

| Operation | Complexity | Notes |
|---|---|---|
| `s[i]` | O(1) | Index access |
| `len(s)` | O(1) | Stored as a field |
| `s + t` | O(n + m) | Allocates new string |
| `s += c` in a loop | **O(n²)** | Builds quadratic-time garbage |
| `''.join(list_of_strings)` | O(total length) | The Pythonic builder |
| `s[i:j]` | O(j - i) | Copies that slice |
| `s == t` | O(n) | Char by char |
| `c in s` | O(n × m) worst | Implemented as substring search |
| `s.find(p)`, `p in s` | O(n × m) worst, near-O(n) average | CPython uses a tuned algorithm |
| `s.replace(a, b)` | O(n + m) | |
| `s.split(sep)` | O(n) | |
| `s.lower()`, `s.upper()` | O(n) | New string |
| `sorted(s)` | O(n log n) | Returns a list of chars |
| `s == s[::-1]` (reverse check) | O(n) | But allocates a full copy |
| `hash(s)` | O(n) first time, O(1) after | Cached in CPython |

## Character codes
- `ord('a')` → 97 — converts char to int
- `chr(97)` → 'a' — converts int to char
- `ord('A')` → 65, `ord('a') - ord('A')` → 32 — useful for case manipulation
- `ord('0')` → 48 — useful when parsing digits with `ord(c) - ord('0')`

## Useful tests on a single character
- `c.isdigit()`, `c.isalpha()`, `c.isalnum()`, `c.isspace()`, `c.isupper()`, `c.islower()`
- These all work on a multi-char string too (returns True iff every char passes), so `"abc".isalpha() == True`.


In [ ]:
# 1.1 Demonstrating immutability + the join trick
import time

n = 10_000

# Bad — O(n^2) concat
t0 = time.perf_counter()
s = ""
for i in range(n):
    s += "a"
t1 = time.perf_counter()

# Good — O(n) join
t2 = time.perf_counter()
parts = []
for i in range(n):
    parts.append("a")
s2 = "".join(parts)
t3 = time.perf_counter()

print(f"+=    : {(t1-t0)*1000:.1f} ms")
print(f"join  : {(t3-t2)*1000:.1f} ms")
# CPython has an optimization for `s += c` in some cases, but you can't rely on it.
# Always reach for join() in interview code.


In [ ]:
# 1.2 ord/chr — the bread and butter of string problems
print(ord('a'), ord('z'), ord('A'), ord('Z'), ord('0'), ord('9'))

# Lowercase <-> uppercase via XOR with 32 (only for ASCII letters)
print(chr(ord('A') ^ 32))     # 'a'
print(chr(ord('z') ^ 32))     # 'Z'

# Build a small alphabet
alphabet = [chr(ord('a') + i) for i in range(26)]
print(alphabet)

# Parse a digit character to int — faster than int(c) in hot loops
c = '7'
print(ord(c) - ord('0'))


In [ ]:
# 1.3 Character classification
print('a'.isalpha(), '7'.isdigit(), 'a7'.isalnum())   # True True True
print(' '.isspace(), 'A'.isupper(), 'a'.islower())     # True True True

# Common cleanup idiom — keep only alphanumerics, lowercased
def normalize(s):
    return ''.join(c.lower() for c in s if c.isalnum())

print(normalize("A man, a plan, a canal: Panama"))     # 'amanaplanacanalpanama'


---
# 2. `str` methods — the reference you need to internalize

There are ~45 string methods. About 15 carry their weight in interviews. Run all the cells below; the rest can be looked up.

In [ ]:
# 2.1 Case conversion
s = "Hello, World!"
print(s.upper())              # HELLO, WORLD!
print(s.lower())              # hello, world!
print(s.swapcase())           # hELLO, wORLD!
print(s.title())              # Hello, World!
print(s.capitalize())         # Hello, world!   — only first char
print("hello world".casefold())   # 'hello world' — aggressive lowercasing for unicode (e.g. German ß -> ss)


In [ ]:
# 2.2 Search & test — return bool or index
s = "Hello, World!"

# IN operator — the simplest test
print("World" in s)               # True
print("xyz" in s)                 # False

# find vs index — both return the position of the first occurrence
print(s.find("World"))            # 7
print(s.find("World", 8))         # -1 — start search at index 8
print(s.find("xyz"))              # -1 — NOT found returns -1
# print(s.index("xyz"))           # raises ValueError — use find() instead in defensive code

# rfind / rindex — search from the RIGHT
print(s.rfind("l"))               # 10 — last 'l'

# count — non-overlapping occurrences
print(s.count("l"))               # 3

# startswith / endswith — accept a tuple of options
print(s.startswith("Hello"))      # True
print(s.endswith(("?", "!", ".")))   # True — useful for sentence-end detection


In [ ]:
# 2.3 Strip / trim
print("   hello   ".strip())             # 'hello'
print("   hello   ".lstrip())            # 'hello   '
print("   hello   ".rstrip())            # '   hello'

# strip accepts a SET of chars to remove (not a substring!)
print("xxhelloxx".strip("x"))            # 'hello'
print("xyxyhelloxyxy".strip("xy"))       # 'hello' — strips any combo of 'x' and 'y'

# common mistake — strip("abc") doesn't remove the substring 'abc'
print("abchelloabc".strip("abc"))        # 'hello'  — removes any leading/trailing a/b/c


In [ ]:
# 2.4 Replace
s = "abcabc"
print(s.replace("a", "X"))           # 'XbcXbc'
print(s.replace("a", "X", 1))        # 'Xbcabc' — count argument: only replace first N
print(s.replace("ab", ""))           # 'cc'    — useful for deletion


In [ ]:
# 2.5 Split / join
s = "a,b,c,,d"
print(s.split(","))                  # ['a', 'b', 'c', '', 'd']
print(s.split(",", 2))               # ['a', 'b', 'c,,d']  — max splits
print(s.rsplit(",", 1))              # ['a,b,c,', 'd']      — split from right

# split() with NO args = split on any whitespace, no empty strings in output
print("  hello   world  ".split())   # ['hello', 'world']

# splitlines — splits on line boundaries
print("line1\nline2\rline3\r\nline4".splitlines())   # ['line1','line2','line3','line4']

# Join — the inverse
print("-".join(["a", "b", "c"]))     # 'a-b-c'
print("".join(["a", "b", "c"]))      # 'abc'


In [ ]:
# 2.6 Padding / alignment
print("42".zfill(5))                 # '00042'   — zero-fill on the left
print("hi".ljust(8, '-'))            # 'hi------'
print("hi".rjust(8, '-'))            # '------hi'
print("hi".center(8, '*'))           # '***hi***'


In [ ]:
# 2.7 f-strings — the modern way to format
name = "Sam"
age = 25
print(f"Hi, {name}. You are {age}.")
print(f"{name=}, {age=}")            # 'name='Sam', age=25'  — debug form (3.8+)

# Width, precision, alignment
pi = 3.14159265
print(f"{pi:.2f}")                   # '3.14'   — 2 decimal places
print(f"{pi:>10.3f}")                # '     3.142'  — right-align, width 10
print(f"{pi:<10.3f}|")               # '3.142     |' — left-align
print(f"{pi:^10.3f}")                # '  3.142   '  — center
print(f"{1_000_000:,}")              # '1,000,000'   — thousands separators
print(f"{255:08b}")                  # '11111111'    — binary, zero-padded to 8
print(f"{255:#x}")                   # '0xff'        — hex with prefix


In [ ]:
# 2.8 Encoding & decoding — when bytes show up
s = "café"
b = s.encode("utf-8")                # str -> bytes
print(b, type(b))                    # b'caf\xc3\xa9'
print(b.decode("utf-8"))             # 'café'
print(len(s), len(b))                # 4, 5 — 'é' is 2 bytes in UTF-8

# Encoding mistakes — common interview trap on internationalized text
# 'ñ' is one CHARACTER but in UTF-8 it's two BYTES, so len() values differ.


In [ ]:
# 2.9 The translate / maketrans trick — bulk char substitution in O(n)
table = str.maketrans("aeiou", "AEIOU")
print("hello world".translate(table))   # 'hEllO wOrld'

# Deletion — third arg of maketrans
table = str.maketrans("", "", "aeiou")
print("hello world".translate(table))   # 'hll wrld'  — vowels removed


---
# 3. Two pointers on strings

**Story:** strings are arrays. Every two-pointer trick from the array notebook applies. The most common string flavours:
- **Opposite ends** — palindrome checks, reverse a string, valid palindrome ignoring non-alphanumerics.
- **Same direction** — in-place compression, deduplication, write-pointer pattern.
- **Anchored expansion** — pick a center, expand outward (expand-around-center for palindromes; see §6).

In [ ]:
# 3.1 Reverse a string — list slice vs in-place
def reverse_str(s):
    return s[::-1]                     # one-liner, allocates a new string

def reverse_chars_inplace(chars):
    '''LC 344 — given a list of chars, reverse in place.'''
    l, r = 0, len(chars) - 1
    while l < r:
        chars[l], chars[r] = chars[r], chars[l]
        l += 1
        r -= 1

print(reverse_str("hello"))
a = list("hello"); reverse_chars_inplace(a); print(''.join(a))


In [ ]:
# 3.2 Valid palindrome (ignore non-alphanumerics, case-insensitive) — LC 125
def is_palindrome(s):
    l, r = 0, len(s) - 1
    while l < r:
        # skip non-alphanumerics on either side
        while l < r and not s[l].isalnum():
            l += 1
        while l < r and not s[r].isalnum():
            r -= 1
        if s[l].lower() != s[r].lower():
            return False
        l += 1
        r -= 1
    return True

assert is_palindrome("A man, a plan, a canal: Panama") == True
assert is_palindrome("race a car") == False
assert is_palindrome(" ") == True
print("is_palindrome OK")


In [ ]:
# 3.3 Valid palindrome II — delete AT MOST one character — LC 680
def valid_palindrome_one_delete(s):
    def is_pal_range(l, r):
        while l < r:
            if s[l] != s[r]: return False
            l += 1; r -= 1
        return True
    l, r = 0, len(s) - 1
    while l < r:
        if s[l] != s[r]:
            # try deleting either side
            return is_pal_range(l + 1, r) or is_pal_range(l, r - 1)
        l += 1; r -= 1
    return True

assert valid_palindrome_one_delete("aba") == True
assert valid_palindrome_one_delete("abca") == True       # delete 'b' or 'c'
assert valid_palindrome_one_delete("abc") == False
print("valid_palindrome_one_delete OK")


In [ ]:
# 3.4 Reverse words in a string — LC 151
def reverse_words(s):
    # The one-line Pythonic answer; interviewers usually want you to do it manually too
    return " ".join(s.split()[::-1])

# Manual O(1) extra space (on a mutable char list) version — show this if asked
def reverse_words_inplace(chars):
    '''chars is a list of single chars. Returns the reversed-words list.'''
    n = len(chars)
    # 1. reverse the whole list
    chars.reverse()
    # 2. reverse each word back
    i = 0
    while i < n:
        j = i
        while j < n and chars[j] != ' ':
            j += 1
        # reverse chars[i:j]
        l, r = i, j - 1
        while l < r:
            chars[l], chars[r] = chars[r], chars[l]
            l += 1; r -= 1
        i = j + 1
    return chars

print(reverse_words("the sky is blue"))           # 'blue is sky the'
print(reverse_words("  hello   world  "))         # 'world hello' — split() ignores spaces


In [ ]:
# 3.5 String compression — write-pointer pattern (in place) — LC 443
def compress(chars):
    '''Replace runs of equal chars with char + count (if >1). Return new length.'''
    write = 0
    read = 0
    n = len(chars)
    while read < n:
        ch = chars[read]
        count = 0
        while read < n and chars[read] == ch:
            read += 1
            count += 1
        chars[write] = ch
        write += 1
        if count > 1:
            for d in str(count):
                chars[write] = d
                write += 1
    return write

a = ["a","a","b","b","c","c","c"]
k = compress(a)
print(k, a[:k])                                   # 6 ['a','2','b','2','c','3']
a = ["a"]; k = compress(a); print(k, a[:k])       # 1 ['a']
a = ["a","b","b","b","b","b","b","b","b","b","b","b","b"]
k = compress(a); print(k, a[:k])                  # 4 ['a','b','1','2']


**Practice — two pointers on strings**

| Concept | LeetCode |
|---|---|
| Reverse string | LC 344 |
| Reverse string II (every 2k group) | LC 541 |
| Reverse words | LC 151 |
| Reverse words in string III (each word) | LC 557 |
| Valid palindrome | LC 125 |
| Valid palindrome II (one deletion) | LC 680 |
| Is subsequence | LC 392 |
| String compression | LC 443 |
| Backspace string compare | LC 844 |
| Reverse vowels | LC 345 |


---
# 4. Sliding window on strings

**Story:** "substring with property X" — longest, shortest, or count. The template from the arrays/hashing notebooks applies directly, but a few string-specific tricks earn their own section.

**Three variable-window flavours you'll see on strings:**
1. **Longest substring satisfying property** — expand `r`, when violated shrink `l`, update `max(best, r - l + 1)` whenever VALID.
2. **Shortest substring satisfying property** — expand `r` until valid, then shrink `l` while still valid; update `min(best, r - l + 1)`.
3. **Count substrings with exact property** — often: (count with ≤ K) − (count with ≤ K-1). Useful for "exactly K distinct chars."

**The character-count map** is your state — use `defaultdict(int)` or a fixed-size `[0]*26` array.

In [ ]:
# 4.1 Longest substring without repeating characters — LC 3
def longest_unique_substring(s):
    last_seen = {}                         # char -> most recent index
    l = 0
    best = 0
    for r, ch in enumerate(s):
        if ch in last_seen and last_seen[ch] >= l:
            l = last_seen[ch] + 1          # jump l past the previous occurrence
        last_seen[ch] = r
        best = max(best, r - l + 1)
    return best

assert longest_unique_substring("abcabcbb") == 3
assert longest_unique_substring("bbbbb") == 1
assert longest_unique_substring("pwwkew") == 3
print("longest_unique_substring OK")


In [ ]:
# 4.2 Longest substring with at most K distinct characters — LC 340
from collections import defaultdict

def longest_k_distinct(s, k):
    freq = defaultdict(int)
    l = 0
    best = 0
    for r, ch in enumerate(s):
        freq[ch] += 1
        while len(freq) > k:
            freq[s[l]] -= 1
            if freq[s[l]] == 0:
                del freq[s[l]]
            l += 1
        best = max(best, r - l + 1)
    return best

assert longest_k_distinct("eceba", 2) == 3
assert longest_k_distinct("aa", 1) == 2
print("longest_k_distinct OK")


In [ ]:
# 4.3 Longest repeating character replacement — LC 424
# Replace at most K chars in a window so that all chars are the same. Find the longest such window.
# Trick: window length - max_freq_in_window <= k
def character_replacement(s, k):
    counts = defaultdict(int)
    l = 0
    best = 0
    max_freq = 0                              # max char count seen in any state
    for r, ch in enumerate(s):
        counts[ch] += 1
        max_freq = max(max_freq, counts[ch])
        # window size = r - l + 1; chars to replace = window - max_freq
        if (r - l + 1) - max_freq > k:
            counts[s[l]] -= 1
            l += 1
        # NOTE: we do NOT update max_freq on shrink — that's the key trick.
        # The best answer is monotone: it can only get bigger, never smaller.
        best = max(best, r - l + 1)
    return best

assert character_replacement("ABAB", 2) == 4
assert character_replacement("AABABBA", 1) == 4
print("character_replacement OK")


In [ ]:
# 4.4 Find all anagrams of p in s — LC 438 (fixed window with frequency match)
from collections import Counter

def find_anagrams(s, p):
    if len(p) > len(s):
        return []
    need = Counter(p)
    window = Counter(s[:len(p)])
    res = []
    if window == need:
        res.append(0)
    for r in range(len(p), len(s)):
        window[s[r]] += 1
        left_char = s[r - len(p)]
        window[left_char] -= 1
        if window[left_char] == 0:
            del window[left_char]
        if window == need:
            res.append(r - len(p) + 1)
    return res

print(find_anagrams("cbaebabacd", "abc"))     # [0, 6]
print(find_anagrams("abab", "ab"))            # [0, 1, 2]


In [ ]:
# 4.5 Permutation in string — LC 567 — same idea, return bool
def check_inclusion(s1, s2):
    if len(s1) > len(s2):
        return False
    need = [0] * 26
    window = [0] * 26
    for c in s1:
        need[ord(c) - ord('a')] += 1
    for i in range(len(s1)):
        window[ord(s2[i]) - ord('a')] += 1
    if window == need:
        return True
    for r in range(len(s1), len(s2)):
        window[ord(s2[r]) - ord('a')] += 1
        window[ord(s2[r - len(s1)]) - ord('a')] -= 1
        if window == need:
            return True
    return False

assert check_inclusion("ab", "eidbaooo") == True
assert check_inclusion("ab", "eidboaoo") == False
print("check_inclusion OK")


In [ ]:
# 4.6 Minimum window substring — LC 76 — THE classic shortest-window
def min_window(s, t):
    if not t or not s:
        return ""
    need = Counter(t)
    missing = len(t)                       # how many chars (with multiplicity) still needed
    l = 0
    start = 0
    best = float('inf')
    for r, ch in enumerate(s):
        if need[ch] > 0:
            missing -= 1
        need[ch] -= 1
        if missing == 0:
            # contract from the left while window still covers t
            while need[s[l]] < 0:
                need[s[l]] += 1
                l += 1
            if r - l + 1 < best:
                best = r - l + 1
                start = l
            # release one needed char to keep searching
            need[s[l]] += 1
            missing += 1
            l += 1
    return "" if best == float('inf') else s[start:start + best]

assert min_window("ADOBECODEBANC", "ABC") == "BANC"
assert min_window("a", "a") == "a"
print("min_window OK")


**Practice — sliding window on strings**

| Concept | LeetCode |
|---|---|
| Longest substring without repeating chars | LC 3 |
| Longest substring with at most K distinct | LC 340 |
| Longest substring with exactly K distinct | LC 992-ish via at-most-K trick |
| Longest repeating character replacement | LC 424 |
| Find all anagrams in a string | LC 438 |
| Permutation in string | LC 567 |
| Minimum window substring | LC 76 |
| Substring with concat of all words | LC 30 |
| Longest substring with at least K repeating chars | LC 395 (window + divide-and-conquer) |
| Fruit into baskets | LC 904 (string variant) |


---
# 5. Character frequency & the anagram family

**Story:** "Are these two strings rearrangements of each other?" Or: "group these strings by the character bag." This entire family is solved with a frequency map.

**Three canonical representations of a string's character bag:**
1. **Sorted string** — `''.join(sorted(s))` — O(k log k) per string
2. **Counter** — `Counter(s)` — O(k)
3. **Tuple of 26 counts** — `tuple(count[i] for i in range(26))` — O(k), fastest, hashable

Tuple-of-counts is the right answer when keys are ASCII letters; Counter wins on readability and works for arbitrary alphabets.

In [ ]:
# 5.1 Valid anagram — LC 242
def is_anagram(s, t):
    if len(s) != len(t):
        return False
    return Counter(s) == Counter(t)

# Faster fixed-array version (lowercase English)
def is_anagram_fast(s, t):
    if len(s) != len(t):
        return False
    counts = [0] * 26
    for cs, ct in zip(s, t):
        counts[ord(cs) - ord('a')] += 1
        counts[ord(ct) - ord('a')] -= 1
    return all(c == 0 for c in counts)

assert is_anagram("anagram", "nagaram") == True
assert is_anagram_fast("rat", "car") == False
print("anagram OK")


In [ ]:
# 5.2 Group anagrams — LC 49 — fast tuple-of-counts key
def group_anagrams(strs):
    groups = defaultdict(list)
    for s in strs:
        counts = [0] * 26
        for ch in s:
            counts[ord(ch) - ord('a')] += 1
        groups[tuple(counts)].append(s)        # tuple is hashable; list is not
    return list(groups.values())

print(group_anagrams(["eat","tea","tan","ate","nat","bat"]))


In [ ]:
# 5.3 Ransom note — LC 383 — can `note` be built from chars of `magazine`?
def can_construct(note, mag):
    cm = Counter(mag)
    for ch in note:
        if cm[ch] <= 0:
            return False
        cm[ch] -= 1
    return True

# One-line version using Counter subtraction
def can_construct_v2(note, mag):
    return not (Counter(note) - Counter(mag))     # Counter - keeps only positive diffs

assert can_construct("aa", "aab") == True
assert can_construct("aa", "ab") == False
assert can_construct_v2("aaa", "aab") == False
print("can_construct OK")


In [ ]:
# 5.4 First non-repeating character — LC 387
def first_uniq_char(s):
    c = Counter(s)
    for i, ch in enumerate(s):
        if c[ch] == 1:
            return i
    return -1

assert first_uniq_char("leetcode") == 0
assert first_uniq_char("loveleetcode") == 2
assert first_uniq_char("aabb") == -1
print("first_uniq_char OK")


In [ ]:
# 5.5 Sort characters by frequency — LC 451
def frequency_sort(s):
    c = Counter(s)
    # most_common returns [(char, count), ...] sorted by count desc
    return ''.join(ch * count for ch, count in c.most_common())

print(frequency_sort("tree"))           # 'eert' or 'eetr' — 'e' first, then 'r'/'t'
print(frequency_sort("Aabb"))           # 'bbAa' or 'bbaA'


**Practice — anagram / frequency**

| Concept | LeetCode |
|---|---|
| Valid anagram | LC 242 |
| Group anagrams | LC 49 |
| Find all anagrams in a string | LC 438 |
| Ransom note | LC 383 |
| First non-repeating character | LC 387 |
| Sort characters by frequency | LC 451 |
| Determine if two strings are close | LC 1657 |
| Find common characters | LC 1002 |
| Custom sort string | LC 791 |


---
# 6. The palindrome family

**Three approaches to palindrome-substring problems, ranked by what you need to know:**

1. **Expand around center** — O(n²) time, O(1) space. THE interview answer. Pick each index as a center; expand outward while characters match. Do it twice per index — once for odd-length (center at the char) and once for even-length (center between two chars).
2. **Dynamic programming** — O(n²) time, O(n²) space. Useful for "count all palindromic substrings" or "longest palindromic subsequence." Lives in the DP notebook.
3. **Manacher's algorithm** — O(n) time. Rarely required in interviews; mention it exists if asked about optimal time. Implementation below for reference.

**Why two expansions per index?** Because palindromes can be either:
- Odd length, with one char in the exact middle: `racecar` (center = 'e')
- Even length, with two chars sharing the middle: `abba` (center between the two 'b's)

If you only expand from one char, you miss every even palindrome.

In [ ]:
# 6.1 Longest palindromic substring — LC 5 — expand around center
def longest_palindrome(s):
    if not s: return ""
    start = 0
    best_len = 1
    def expand(l, r):
        while l >= 0 and r < len(s) and s[l] == s[r]:
            l -= 1; r += 1
        return l + 1, r - 1     # the last MATCHING l and r
    for i in range(len(s)):
        # odd-length palindrome centered at i
        l1, r1 = expand(i, i)
        if r1 - l1 + 1 > best_len:
            best_len = r1 - l1 + 1
            start = l1
        # even-length palindrome centered between i and i+1
        l2, r2 = expand(i, i + 1)
        if r2 - l2 + 1 > best_len:
            best_len = r2 - l2 + 1
            start = l2
    return s[start:start + best_len]

assert longest_palindrome("babad") in ("bab", "aba")
assert longest_palindrome("cbbd") == "bb"
assert longest_palindrome("a") == "a"
print("longest_palindrome OK")


In [ ]:
# 6.2 Count palindromic substrings — LC 647 — same expand-around-center
def count_palindromic_substrings(s):
    count = 0
    def expand(l, r):
        nonlocal count
        while l >= 0 and r < len(s) and s[l] == s[r]:
            count += 1
            l -= 1; r += 1
    for i in range(len(s)):
        expand(i, i)         # odd
        expand(i, i + 1)     # even
    return count

assert count_palindromic_substrings("abc") == 3           # a, b, c
assert count_palindromic_substrings("aaa") == 6           # a×3, aa×2, aaa
print("count_palindromic_substrings OK")


In [ ]:
# 6.3 Manacher's algorithm — O(n) longest palindromic substring
# Worth knowing exists. Implementation here for reference; few interviews require it.
def manacher_longest_palindrome(s):
    if not s: return ""
    # Transform: insert sentinels so we only have to consider odd-length palindromes
    t = '^#' + '#'.join(s) + '#$'
    n = len(t)
    p = [0] * n            # p[i] = radius of palindrome centered at i in transformed string
    c = r = 0              # center and right edge of the rightmost palindrome found
    for i in range(1, n - 1):
        mirror = 2 * c - i
        if i < r:
            p[i] = min(r - i, p[mirror])
        # try to expand
        while t[i + p[i] + 1] == t[i - p[i] - 1]:
            p[i] += 1
        if i + p[i] > r:
            c, r = i, i + p[i]
    # find max radius
    max_r, idx = max((p[i], i) for i in range(n))
    start = (idx - max_r) // 2     # map back to original
    return s[start:start + max_r]

print(manacher_longest_palindrome("babad"))     # 'bab' or 'aba'
print(manacher_longest_palindrome("cbbd"))      # 'bb'
print(manacher_longest_palindrome("forgeeksskeegfor"))   # 'geeksskeeg'


**Practice — palindromes**

| Concept | LeetCode |
|---|---|
| Valid palindrome | LC 125 |
| Valid palindrome II | LC 680 |
| Longest palindromic substring | LC 5 |
| Palindromic substrings count | LC 647 |
| Shortest palindrome (KMP trick — see §7) | LC 214 |
| Palindrome pairs | LC 336 |
| Longest palindrome (you can rearrange chars) | LC 409 |
| Palindrome number | LC 9 |


---
# 7. Pattern matching — KMP & Rabin-Karp

**Story:** find all occurrences of pattern `p` (length m) in text `s` (length n). Naive search is O(n × m). Two algorithms get you to O(n + m):

| Algorithm | Time | Space | Best for |
|---|---|---|---|
| **Naive** | O(n × m) | O(1) | Small inputs, always-correct fallback |
| **KMP** | O(n + m) | O(m) | Deterministic linear; single pattern |
| **Rabin-Karp** | O(n + m) average, O(n × m) worst | O(1) | Multiple pattern queries; rolling-hash trick |
| **Z-function** | O(n + m) | O(n + m) | Prefix-match queries (see §11) |

In an interview, you should be able to write KMP from scratch and explain Rabin-Karp's rolling hash. Most interviewers will accept `s.find(p)` if they only want the *idea* — but if they explicitly say "implement substring search efficiently," they want one of these.

## 7a. KMP — Knuth-Morris-Pratt

KMP avoids rescanning text by precomputing the **LPS (Longest Proper Prefix that is also a Suffix)** array of the pattern. When a mismatch happens after matching `j` characters, instead of restarting at the next text index, we slide the pattern forward by `j - lps[j-1]`.

**LPS — what it means**

`lps[i]` = length of the longest proper prefix of `p[0..i]` that is also a suffix of `p[0..i]`.

| pattern | lps |
|---|---|
| `abcd` | `[0,0,0,0]` |
| `aaaa` | `[0,1,2,3]` |
| `ababc` | `[0,0,1,2,0]` |
| `ababab` | `[0,0,1,2,3,4]` |

In [ ]:
# 7a.1 LPS array — preprocessing for KMP
def compute_lps(p):
    lps = [0] * len(p)
    length = 0      # length of the previous longest prefix that's also a suffix
    i = 1
    while i < len(p):
        if p[i] == p[length]:
            length += 1
            lps[i] = length
            i += 1
        else:
            if length != 0:
                length = lps[length - 1]    # fall back — the key insight
            else:
                lps[i] = 0
                i += 1
    return lps

# Sanity checks against your reference file
assert compute_lps("ababc") == [0, 0, 1, 2, 0]
assert compute_lps("abcd") == [0, 0, 0, 0]
assert compute_lps("ababab") == [0, 0, 1, 2, 3, 4]
assert compute_lps("aaaa") == [0, 1, 2, 3]
print("compute_lps OK")


In [ ]:
# 7a.2 KMP search — returns all match start indices
def kmp_search(text, pattern):
    n, m = len(text), len(pattern)
    if m == 0: return [0]
    lps = compute_lps(pattern)
    res = []
    i = j = 0
    while i < n:
        if text[i] == pattern[j]:
            i += 1; j += 1
            if j == m:
                res.append(i - j)
                j = lps[j - 1]                  # continue looking for more matches
        else:
            if j != 0:
                j = lps[j - 1]                  # slide pattern, do NOT advance text
            else:
                i += 1
    return res

assert kmp_search("abcdefg", "bcd") == [1]
assert kmp_search("aaaaab", "aaaa") == [0, 1]
assert kmp_search("abcd", "ba") == []
assert kmp_search("ababababab", "abab") == [0, 2, 4, 6]
print("kmp_search OK")


## 7b. Rabin-Karp — rolling hash

**Idea:** treat each substring as a number in some base (e.g., 26 or 256), hash it, and slide. When the window moves by one, you can update the hash in O(1) by removing the contribution of the leftmost char and adding the new rightmost char.

**Why a modulus?** To keep numbers bounded. A large prime like `10^9 + 7` minimizes collisions.

**When hash matches, verify char-by-char** to handle collisions. With good prime choice, expected time is O(n + m); pathological adversarial inputs can push it to O(n × m).

In [ ]:
# 7b.1 Rabin-Karp — single pattern search with rolling hash
def rabin_karp(text, pattern, base=256, mod=10**9 + 7):
    n, m = len(text), len(pattern)
    if m == 0: return [0]
    if m > n: return []
    high = pow(base, m - 1, mod)            # base^(m-1) mod p — precompute for the rolling step
    p_hash = 0
    t_hash = 0
    for i in range(m):
        p_hash = (p_hash * base + ord(pattern[i])) % mod
        t_hash = (t_hash * base + ord(text[i]))    % mod
    res = []
    for i in range(n - m + 1):
        if p_hash == t_hash:
            # verify char-by-char in case of hash collision
            if text[i:i + m] == pattern:
                res.append(i)
        if i < n - m:
            # roll the window: drop text[i], add text[i+m]
            t_hash = (t_hash - ord(text[i]) * high) % mod
            t_hash = (t_hash * base + ord(text[i + m])) % mod
            t_hash %= mod
    return res

assert rabin_karp("abcdefg", "bcd") == [1]
assert rabin_karp("aaaaab", "aaaa") == [0, 1]
assert rabin_karp("abcd", "ba") == []
print("rabin_karp OK")


In [ ]:
# 7b.2 When to reach for Rabin-Karp — multiple patterns of the same length
# Or "find any duplicate substring of length L" — classic LC 1044 use case
def repeated_substring_of_length_L(s, L, base=26, mod=10**9 + 7):
    '''Return the START INDEX of any repeated substring of length L, or -1.'''
    if L == 0 or L > len(s): return -1
    high = pow(base, L - 1, mod)
    h = 0
    for i in range(L):
        h = (h * base + (ord(s[i]) - ord('a'))) % mod
    seen = {h: 0}
    for i in range(1, len(s) - L + 1):
        h = (h - (ord(s[i - 1]) - ord('a')) * high) % mod
        h = (h * base + (ord(s[i + L - 1]) - ord('a'))) % mod
        h %= mod
        if h in seen:
            # verify (collision safety)
            j = seen[h]
            if s[j:j + L] == s[i:i + L]:
                return i
        else:
            seen[h] = i
    return -1

print(repeated_substring_of_length_L("banana", 2))      # index of 'an' or 'na'
print(repeated_substring_of_length_L("abcdef", 2))      # -1


**Traps**
- **KMP off-by-one in LPS construction** — the inner `length = lps[length-1]` rather than just `length -= 1` is the part most candidates fumble.
- **Rabin-Karp without char-by-char verification** is incorrect — hash collisions WILL happen with adversarial inputs.
- Forgetting to take `% mod` after the subtraction (which can go negative in Python ... actually Python's `%` returns non-negative, so this is more a C++/Java issue, but it's worth knowing).

**Practice — pattern matching**

| Concept | LeetCode |
|---|---|
| Implement strStr() (substring search) | LC 28 |
| Repeated substring pattern | LC 459 — solvable with KMP's LPS! |
| Shortest palindrome | LC 214 — KMP trick: longest palindrome prefix |
| Longest happy prefix | LC 1392 — direct LPS application |
| Repeated DNA sequences | LC 187 — Rabin-Karp use case |
| Longest duplicate substring | LC 1044 — Rabin-Karp + binary search |
| Find and replace pattern | LC 890 |


---
# 8. String parsing & manipulation

**Story:** "convert this string to a number" / "convert this number to a string" / "validate this format." These problems test whether you handle edge cases — overflow, signs, leading zeros, whitespace, invalid characters.

The art here is **decomposing the problem into stages**: skip whitespace → optional sign → digits → ignore trailing junk. Stage-by-stage, each stage is trivial; what trips candidates up is mashing them together.

In [ ]:
# 8.1 atoi — string to integer — LC 8
def my_atoi(s):
    INT_MAX = 2**31 - 1
    INT_MIN = -2**31
    i = 0
    n = len(s)
    # 1. skip leading whitespace
    while i < n and s[i] == ' ':
        i += 1
    # 2. optional sign
    sign = 1
    if i < n and s[i] in ('+', '-'):
        if s[i] == '-':
            sign = -1
        i += 1
    # 3. accumulate digits
    result = 0
    while i < n and s[i].isdigit():
        result = result * 10 + (ord(s[i]) - ord('0'))
        i += 1
    result *= sign
    # 4. clamp to 32-bit signed range
    return max(INT_MIN, min(INT_MAX, result))

assert my_atoi("42") == 42
assert my_atoi("   -42") == -42
assert my_atoi("4193 with words") == 4193
assert my_atoi("words and 987") == 0
assert my_atoi("-91283472332") == -2**31      # clamped
print("my_atoi OK")


In [ ]:
# 8.2 Roman to integer — LC 13
def roman_to_int(s):
    vals = {'I': 1, 'V': 5, 'X': 10, 'L': 50, 'C': 100, 'D': 500, 'M': 1000}
    total = 0
    for i, ch in enumerate(s):
        v = vals[ch]
        # if the NEXT numeral is bigger, this one is being SUBTRACTED (IV, IX, XL, XC, CD, CM)
        if i + 1 < len(s) and vals[s[i + 1]] > v:
            total -= v
        else:
            total += v
    return total

assert roman_to_int("III") == 3
assert roman_to_int("IV") == 4
assert roman_to_int("LVIII") == 58
assert roman_to_int("MCMXCIV") == 1994
print("roman_to_int OK")


In [ ]:
# 8.3 Integer to Roman — LC 12 — greedy with descending values
def int_to_roman(num):
    pairs = [
        (1000, 'M'), (900, 'CM'), (500, 'D'), (400, 'CD'),
        (100, 'C'),  (90, 'XC'),  (50, 'L'),  (40, 'XL'),
        (10, 'X'),   (9, 'IX'),   (5, 'V'),   (4, 'IV'), (1, 'I')
    ]
    res = []
    for v, sym in pairs:
        while num >= v:
            res.append(sym)
            num -= v
    return ''.join(res)

assert int_to_roman(3) == "III"
assert int_to_roman(1994) == "MCMXCIV"
print("int_to_roman OK")


In [ ]:
# 8.4 Add binary strings — LC 67 — without using int(s, 2)
def add_binary(a, b):
    i, j = len(a) - 1, len(b) - 1
    carry = 0
    out = []
    while i >= 0 or j >= 0 or carry:
        x = int(a[i]) if i >= 0 else 0
        y = int(b[j]) if j >= 0 else 0
        total = x + y + carry
        out.append(str(total % 2))
        carry = total // 2
        i -= 1
        j -= 1
    return ''.join(reversed(out))

assert add_binary("11", "1") == "100"
assert add_binary("1010", "1011") == "10101"
print("add_binary OK")


In [ ]:
# 8.5 Multiply strings — LC 43 — schoolbook multiplication
def multiply_strings(num1, num2):
    if num1 == "0" or num2 == "0":
        return "0"
    n, m = len(num1), len(num2)
    result = [0] * (n + m)
    for i in range(n - 1, -1, -1):
        for j in range(m - 1, -1, -1):
            prod = (ord(num1[i]) - ord('0')) * (ord(num2[j]) - ord('0'))
            total = prod + result[i + j + 1]
            result[i + j + 1] = total % 10
            result[i + j] += total // 10
    # strip leading zeros
    s = ''.join(map(str, result)).lstrip('0')
    return s if s else "0"

assert multiply_strings("2", "3") == "6"
assert multiply_strings("123", "456") == "56088"
print("multiply_strings OK")


In [ ]:
# 8.6 Validate IP address — LC 468
def valid_ip(ip):
    def is_ipv4(s):
        parts = s.split('.')
        if len(parts) != 4: return False
        for p in parts:
            if not p or len(p) > 3: return False
            if not p.isdigit(): return False
            if p[0] == '0' and len(p) > 1: return False     # no leading zeros
            if int(p) > 255: return False
        return True
    def is_ipv6(s):
        parts = s.split(':')
        if len(parts) != 8: return False
        for p in parts:
            if not 1 <= len(p) <= 4: return False
            if not all(c in '0123456789abcdefABCDEF' for c in p): return False
        return True
    if is_ipv4(ip): return "IPv4"
    if is_ipv6(ip): return "IPv6"
    return "Neither"

assert valid_ip("172.16.254.1") == "IPv4"
assert valid_ip("2001:0db8:85a3:0000:0000:8a2e:0370:7334") == "IPv6"
assert valid_ip("256.256.256.256") == "Neither"
assert valid_ip("172.16.254.01") == "Neither"           # leading zero
print("valid_ip OK")


**Practice — parsing / manipulation**

| Concept | LeetCode |
|---|---|
| String to integer (atoi) | LC 8 |
| Roman to integer | LC 13 |
| Integer to Roman | LC 12 |
| Add binary | LC 67 |
| Add strings | LC 415 |
| Multiply strings | LC 43 |
| Valid IP address | LC 468 |
| Restore IP addresses | LC 93 (backtracking — outside scope) |
| Number of valid words | LC 1180 |
| Compare version numbers | LC 165 |
| Integer to English words | LC 273 (hard, very classic) |


---
# 9. Encoding & decoding patterns

**Story:** "serialize this list of strings into one string so it can be deserialized unambiguously" / "compress this run of identical chars" / "decode this encoded string with nesting." Two interview-favorite problems:

In [ ]:
# 9.1 Encode and decode strings — LC 271 — length-prefix encoding
def encode(strs):
    '''Encode list of arbitrary strings into a single string.'''
    return ''.join(f"{len(s)}#{s}" for s in strs)

def decode(s):
    res = []
    i = 0
    while i < len(s):
        # read length up to the '#'
        j = i
        while s[j] != '#':
            j += 1
        length = int(s[i:j])
        res.append(s[j + 1: j + 1 + length])
        i = j + 1 + length
    return res

original = ["hello", "world", "", "with#hash", "and digits 123"]
encoded = encode(original)
print(encoded)
print(decode(encoded))
assert decode(encode(original)) == original
print("encode/decode OK")


In [ ]:
# 9.2 String compression (run-length) — LC 443 revisited
# Already covered in §3. The "encoding" framing matters because the inverse problem
# (DECODE a run-length string) is a stack-of-counts problem — see below.

# 9.3 Decode string — LC 394 — nested encoding like "3[a2[c]]" -> "accaccacc"
def decode_string(s):
    stack = []      # stack of (prev_str, repeat_count)
    cur = ""
    k = 0
    for ch in s:
        if ch.isdigit():
            k = k * 10 + int(ch)
        elif ch == '[':
            stack.append((cur, k))      # save what we had so far and the repeat count
            cur, k = "", 0
        elif ch == ']':
            prev, count = stack.pop()
            cur = prev + cur * count
        else:
            cur += ch
    return cur

assert decode_string("3[a]2[bc]") == "aaabcbc"
assert decode_string("3[a2[c]]") == "accaccacc"
assert decode_string("2[abc]3[cd]ef") == "abcabccdcdcdef"
print("decode_string OK")


**Practice — encoding/decoding**

| Concept | LeetCode |
|---|---|
| Encode and decode strings | LC 271 |
| String compression | LC 443 |
| Decode string (nested) | LC 394 |
| Run-length encoding (basic) | (variant) |
| Count and say | LC 38 |
| Serialize/deserialize binary tree (tree side) | LC 297 |
| URL shortener encode/decode | LC 535 |


---
# 10. Trie / prefix tree

**Story:** when problems involve **many words** and **prefix queries** ("does any word start with...", "autocomplete", "find longest common prefix"), a trie crushes hashmap-based approaches because lookups are O(L) in word length, not O(L × n) for n words.

**Structure:** each node has a dict of `char -> child node` plus a flag `is_end`. Insertion / search / startswith all walk char-by-char from the root.

In [ ]:
# 10.1 Trie — LC 208 implementation
class TrieNode:
    __slots__ = ('children', 'is_end')
    def __init__(self):
        self.children = {}
        self.is_end = False

class Trie:
    def __init__(self):
        self.root = TrieNode()

    def insert(self, word):
        node = self.root
        for ch in word:
            if ch not in node.children:
                node.children[ch] = TrieNode()
            node = node.children[ch]
        node.is_end = True

    def search(self, word):
        node = self._walk(word)
        return node is not None and node.is_end

    def startsWith(self, prefix):
        return self._walk(prefix) is not None

    def _walk(self, s):
        node = self.root
        for ch in s:
            if ch not in node.children:
                return None
            node = node.children[ch]
        return node

t = Trie()
for w in ["apple", "app", "apricot"]:
    t.insert(w)
print(t.search("apple"))         # True
print(t.search("app"))           # True
print(t.search("ap"))            # False — not a full word
print(t.startsWith("ap"))        # True
print(t.startsWith("ban"))       # False


In [ ]:
# 10.2 Longest common prefix — LC 14 — character-by-character (no trie needed)
def longest_common_prefix(strs):
    if not strs: return ""
    for i, ch in enumerate(strs[0]):
        for w in strs[1:]:
            if i >= len(w) or w[i] != ch:
                return strs[0][:i]
    return strs[0]

assert longest_common_prefix(["flower","flow","flight"]) == "fl"
assert longest_common_prefix(["dog","racecar","car"]) == ""
print("longest_common_prefix OK")


In [ ]:
# 10.3 Word dictionary with '.' wildcards — LC 211 — DFS into the trie
class WordDictionary:
    def __init__(self):
        self.root = TrieNode()

    def addWord(self, word):
        node = self.root
        for ch in word:
            node = node.children.setdefault(ch, TrieNode())
        node.is_end = True

    def search(self, word):
        def dfs(node, i):
            if i == len(word):
                return node.is_end
            ch = word[i]
            if ch == '.':
                return any(dfs(child, i + 1) for child in node.children.values())
            else:
                return ch in node.children and dfs(node.children[ch], i + 1)
        return dfs(self.root, 0)

wd = WordDictionary()
wd.addWord("bad"); wd.addWord("dad"); wd.addWord("mad")
print(wd.search("pad"))        # False
print(wd.search("bad"))        # True
print(wd.search(".ad"))        # True
print(wd.search("b.."))        # True


**Practice — tries**

| Concept | LeetCode |
|---|---|
| Implement Trie (Prefix Tree) | LC 208 |
| Add and search word — wildcards | LC 211 |
| Longest common prefix | LC 14 |
| Replace words (root replacement using trie) | LC 648 |
| Word break (trie + DP) | LC 139 |
| Concatenated words | LC 472 |
| Word search II (trie + DFS on grid) | LC 212 |
| Stream of characters | LC 1032 — reverse trie trick |
| Maximum XOR of two numbers (bit trie) | LC 421 |
| Top K frequent words | LC 692 (heap or trie) |


---
# 11. The Z-function — brief, often-skippable

**Story:** for each index `i`, the Z-function gives the length of the longest substring starting at `i` that **matches a prefix** of the string. This is useful for prefix queries and is an alternative to KMP's LPS array.

You usually don't need this in interviews — but knowing it exists, and being able to derive how it solves pattern matching, signals strong fundamentals to a Google-tier interviewer.

**Pattern matching trick:** to find all occurrences of `pattern` in `text`, compute the Z-function of `pattern + "$" + text` (where `$` is a separator not present in either). Any index `i > len(pattern)` with `Z[i] == len(pattern)` is a match position (offset by `len(pattern) + 1`).

In [ ]:
# 11.1 Z-function
def z_function(s):
    n = len(s)
    z = [0] * n
    z[0] = n
    l = r = 0
    for i in range(1, n):
        if i < r:
            z[i] = min(r - i, z[i - l])
        while i + z[i] < n and s[z[i]] == s[i + z[i]]:
            z[i] += 1
        if i + z[i] > r:
            l, r = i, i + z[i]
    return z

print(z_function("aabxaayaab"))        # [10, 1, 0, 0, 2, 1, 0, 3, 1, 0]
print(z_function("aaaa"))              # [4, 3, 2, 1]


In [ ]:
# 11.2 Pattern matching via Z-function
def z_search(text, pattern):
    sep = '\x00'
    z = z_function(pattern + sep + text)
    m = len(pattern)
    res = []
    for i in range(m + 1, len(z)):
        if z[i] >= m:
            res.append(i - m - 1)
    return res

assert z_search("abcdefg", "bcd") == [1]
assert z_search("aaaaab", "aaaa") == [0, 1]
print("z_search OK")


---
# 12. Lexicographic & permutation problems

**Story:** "next permutation," "rank of a string," "lexicographically smallest after K swaps." These problems hinge on understanding the **lexicographic order** (dictionary order) and the math of permutations.

Key facts:
- The total number of permutations of `n` distinct characters is `n!`.
- The number of permutations of an n-char string starting with characters less than `c` is `(count_of_chars_less_than_c) * (n-1)!`, adjusted for duplicates.
- The `next permutation` algorithm runs in O(n) and is the basis for `itertools.permutations`.

In [ ]:
# 12.1 Next permutation (in place) — LC 31
def next_permutation(nums):
    n = len(nums)
    # 1. find first decreasing element from the right — the "pivot"
    i = n - 2
    while i >= 0 and nums[i] >= nums[i + 1]:
        i -= 1
    if i >= 0:
        # 2. find rightmost element greater than pivot, swap them
        j = n - 1
        while nums[j] <= nums[i]:
            j -= 1
        nums[i], nums[j] = nums[j], nums[i]
    # 3. reverse the suffix after i — turns "next-bigger" into "smallest next-bigger"
    l, r = i + 1, n - 1
    while l < r:
        nums[l], nums[r] = nums[r], nums[l]
        l += 1; r -= 1

a = [1, 2, 3]; next_permutation(a); print(a)         # [1, 3, 2]
a = [3, 2, 1]; next_permutation(a); print(a)         # [1, 2, 3] — wraps around
a = [1, 1, 5]; next_permutation(a); print(a)         # [1, 5, 1]


In [ ]:
# 12.2 Lexicographic rank of a string (assuming distinct characters) — your reference's problem
# rank = 1 + sum over each prefix position of:
#   (chars smaller than this one, still available) * (factorial of remaining length)
import math

def lex_rank(s):
    n = len(s)
    rank = 1
    used = [False] * 256
    fact = math.factorial(n - 1)
    for i, ch in enumerate(s):
        # count chars smaller than `ch` that haven't been used yet
        smaller = 0
        for c in range(ord(ch)):
            if c < 256 and not used[c]:
                # only count chars that actually exist in s — for the distinct-chars case
                # we still need them to be in s; assume all chars in s are present
                pass
        # cleaner version when alphabet is just chars in s:
        smaller = sum(1 for j in range(i + 1, n) if s[j] < ch)
        rank += smaller * fact
        used[ord(ch)] = True
        if n - 1 - i > 0:
            fact //= (n - 1 - i)
    return rank

# Sanity checks against your reference file
assert lex_rank("abcd") == 1
assert lex_rank("bac") == 3
assert lex_rank("cba") == 6
assert lex_rank("dcba") == 24
print("lex_rank OK")


**Practice — lex / permutation**

| Concept | LeetCode |
|---|---|
| Next permutation | LC 31 |
| Previous permutation | LC 1053 |
| Permutation rank | (GFG classic) |
| Permutation at index K | LC 60 — Permutation Sequence |
| Lexicographically smallest after K swaps | (variant) |
| Smallest subsequence of distinct chars | LC 1081 |
| Remove duplicate letters | LC 316 |
| Largest number (custom comparator) | LC 179 |


---
# 13. Stack on strings — parentheses & nested structures

**Story:** anything with brackets, nested expressions, or "match the most recent open with this close" is a stack pattern. Lives at the intersection of strings and stacks; worth a section here since the problems show up in string interviews.

In [ ]:
# 13.1 Valid parentheses — LC 20
def is_valid(s):
    pairs = {')': '(', ']': '[', '}': '{'}
    stack = []
    for ch in s:
        if ch in '([{':
            stack.append(ch)
        else:
            if not stack or stack[-1] != pairs[ch]:
                return False
            stack.pop()
    return not stack

assert is_valid("()") == True
assert is_valid("()[]{}") == True
assert is_valid("(]") == False
assert is_valid("([)]") == False
print("is_valid OK")


In [ ]:
# 13.2 Remove all adjacent duplicates — LC 1047
def remove_adjacent_duplicates(s):
    stack = []
    for ch in s:
        if stack and stack[-1] == ch:
            stack.pop()
        else:
            stack.append(ch)
    return ''.join(stack)

assert remove_adjacent_duplicates("abbaca") == "ca"
print("remove_adjacent_duplicates OK")


In [ ]:
# 13.3 Simplify path — LC 71 — Unix-style path normalization
def simplify_path(path):
    stack = []
    for token in path.split('/'):
        if token == '' or token == '.':
            continue
        elif token == '..':
            if stack:
                stack.pop()
        else:
            stack.append(token)
    return '/' + '/'.join(stack)

assert simplify_path("/home/") == "/home"
assert simplify_path("/../") == "/"
assert simplify_path("/home//foo/") == "/home/foo"
assert simplify_path("/a/./b/../../c/") == "/c"
print("simplify_path OK")


**Practice — stack on strings**

| Concept | LeetCode |
|---|---|
| Valid parentheses | LC 20 |
| Generate parentheses | LC 22 (backtracking — outside scope, but related) |
| Longest valid parentheses | LC 32 |
| Remove all adjacent duplicates | LC 1047 |
| Remove all adjacent duplicates in string II | LC 1209 |
| Simplify path | LC 71 |
| Evaluate Reverse Polish Notation | LC 150 |
| Basic calculator I/II/III | LC 224 / LC 227 / LC 772 |
| Asteroid collision | LC 735 |
| Minimum remove to make valid parentheses | LC 1249 |


---
# 14. Pattern-recognition cheat sheet

| Problem signal | Reach for |
|---|---|
| "Is X a palindrome?" / "Reverse X" / "Compare from both ends" | Two pointers (opposite ends) |
| "In-place modification / write-pointer" | Two pointers (same direction) |
| "Substring with property X" (longest / shortest / count) | Sliding window + frequency map |
| "Anagrams" / "rearrangement of each other" | Frequency map / tuple-of-counts |
| "Longest palindromic substring/count" | Expand around center, 2 expansions per index |
| "Find all occurrences of pattern" | KMP (deterministic) or Rabin-Karp (rolling hash for variants) |
| "Repeated substring of length L" | Rabin-Karp + hashmap |
| "Prefix queries, autocomplete, word dictionary" | Trie |
| "Wildcard / regex-like search over a dictionary of words" | Trie + DFS |
| "Parse this format (atoi, IP, Roman, version)" | State machine: skip whitespace → sign → digits → ignore junk |
| "Decode nested string like `3[a2[c]]`" | Stack of (prefix, repeat) |
| "Balanced parentheses / match nested" | Stack |
| "Next permutation / lex rank / kth permutation" | Pivot-and-reverse / factorial decomposition |
| "Encode list of strings unambiguously" | Length-prefix encoding |

---

# 15. Interview narration on string problems

Before writing code, narrate:

1. **Character set.** "Is this lowercase English only? ASCII? Unicode?" The answer dictates whether you can use `[0]*26` or need a dict.
2. **Case sensitivity.** "Should 'A' and 'a' match?"
3. **Whitespace / punctuation.** "Do we ignore non-alphanumerics?"
4. **Empty / single char edge cases.** Most string-problem bugs live here.
5. **Why is the brute force slow?** "I could check every substring — that's O(n²) substrings, each tested in O(m), so O(n²·m). With a sliding window the work per character is amortized constant, dropping us to O(n)."
6. **What's the state?** "I'll maintain a frequency map of the current window. When the window violates the property, I shrink from the left."
7. **Building strings carefully.** "I'll collect into a list and join at the end — concatenating strings in a loop is O(n²)."

When asked the "**why is `s += c` quadratic?**" question:
> "Strings in Python are immutable, so each `+=` allocates a new string and copies the old contents over. After n appends you've done 1+2+...+n = O(n²) character copies. The fix is to collect into a list and use `''.join(list)` at the end — that single allocation is O(n)."

When asked about **two pointers vs sliding window**:
> "Two pointers covers any technique that walks two indices through the string — they can move in opposite directions (palindrome check) or the same direction at different speeds. Sliding window is a SPECIFIC case of same-direction two pointers where the indices define the bounds of a contiguous window and we maintain some aggregate state — usually a frequency map — over that window. So sliding window is a strict subset of two pointers."

---

# 16. Common bugs in string problems

1. **`s += c` in a loop.** O(n²). Use `list.append` + `''.join`.
2. **Forgetting empty string / single char cases.** `is_palindrome("")` should be True for most definitions.
3. **Mixing string `find` and `index`.** `find` returns -1 on miss; `index` raises ValueError.
4. **`s.strip("abc")` thinking it removes the substring "abc".** It removes any combination of a/b/c from the ends.
5. **Comparing string length to byte length.** `len("café") == 4`; `len("café".encode("utf-8")) == 5`. Matters for I/O and network problems.
6. **Slicing instead of two-pointer iteration.** `s[i:j]` creates a copy — O(j-i). Inside a loop, this can sneak in a quadratic factor.
7. **Sliding window `max_freq` reset on shrink.** In LC 424, you should NOT update `max_freq` when shrinking — the answer is monotone.
8. **Off-by-one in expand-around-center.** The function returns the last MATCHING `l` and `r`; the length is `r - l + 1`, not `r - l`.
9. **Rabin-Karp without hash collision verification.** Always verify char-by-char on a hash match. Without it, your solution is wrong on adversarial inputs.
10. **KMP LPS — naïve "go back by 1 on mismatch."** The fallback is `length = lps[length - 1]`, not `length -= 1`. Getting this wrong gives O(n × m) worst-case behaviour.
11. **Forgetting both odd and even centers for palindromes.** Two expansions per index.
12. **Letting Python's `int` save you on overflow problems.** In LC 8 (atoi), you have to manually clamp to 32-bit range — Python ints never overflow, so the bug isn't visible until the grader runs.
